In [1]:
import numpy as np
import pandas as pd
import json

np.random.seed(33)

# Sumber 1: Transaksi (CSV) — 5000 baris
n_trx = 5000
kategori_list = ["Elektronik", "Fashion", "Makanan", "Rumah Tangga", "Kesehatan"]
df_trx_tugas6 = pd.DataFrame({
    "order_id": [f"TX{i}" for i in range(n_trx)],
    "product_id": np.random.randint(1, 31, size=n_trx),
    "unit_terjual": np.random.randint(1, 8, size=n_trx),
    "tanggal": np.random.choice(pd.date_range("2026-10-01","2026-10-31"), size=n_trx).astype(str),
})
df_trx_tugas6.to_csv("tugas6_transaksi.csv", index=False)

# Sumber 2: Data produk (JSON Lines) — 30 produk
produk = [
    {"product_id": i, "nama_produk": f"Produk-{i}", "kategori": np.random.choice(kategori_list),
     "harga": int(np.random.choice([25000,50000,75000,100000,150000,250000]))}
    for i in range(1, 31)
]
with open("tugas6_produk.json", "w") as f:
    for p in produk:
        f.write(json.dumps(p) + "\n")

# Sumber 3: Data ulasan (CSV) — tidak semua transaksi memiliki ulasan (realistis)
n_review = 3500
df_review = pd.DataFrame({
    "order_id": np.random.choice(df_trx_tugas6["order_id"], size=n_review, replace=False),
    "rating": np.random.randint(1, 6, size=n_review),
})
df_review.to_csv("tugas6_ulasan.csv", index=False)

print(f"Tiga sumber data berhasil dibuat:")
print(f"- tugas6_transaksi.csv : {len(df_trx_tugas6)} baris")
print(f"- tugas6_produk.json   : {len(produk)} baris")
print(f"- tugas6_ulasan.csv    : {len(df_review)} baris (tidak seluruh transaksi memiliki ulasan)")

Tiga sumber data berhasil dibuat:
- tugas6_transaksi.csv : 5000 baris
- tugas6_produk.json   : 30 baris
- tugas6_ulasan.csv    : 3500 baris (tidak seluruh transaksi memiliki ulasan)


In [5]:
# Impor library yang dibutuhkan
from pyspark.sql import SparkSession

# Membuat SparkSession untuk Tugas Mandiri
spark = SparkSession.builder \
    .appName("Tugas6_2505060006_VandraNoprianusSembiring") \
    .master("local[*]") \
    .getOrCreate()

# Mengurangi log yang tidak perlu agar tampilan lebih rapi
spark.sparkContext.setLogLevel("ERROR")

print("SparkSession untuk Tugas Mandiri siap!")
print("Versi Spark:", spark.version)

SparkSession untuk Tugas Mandiri siap!
Versi Spark: 3.5.9


In [7]:
# A. EXTRACT
# 1. Membaca Transaksi (CSV)
df_transaksi = spark.read.csv("tugas6_transaksi.csv", header=True, inferSchema=True)
print("Data Transaksi:", df_transaksi.count(), "baris")
df_transaksi.printSchema()

# 2. Membaca Produk (JSON)
df_produk = spark.read.json("tugas6_produk.json")
print("Data Produk:", df_produk.count(), "baris")
df_produk.printSchema()

# 3. Membaca Ulasan (CSV)
df_ulasan = spark.read.csv("tugas6_ulasan.csv", header=True, inferSchema=True)
print("Data Ulasan:", df_ulasan.count(), "baris")
df_ulasan.printSchema()


Data Transaksi: 5000 baris
root
 |-- order_id: string (nullable = true)
 |-- product_id: integer (nullable = true)
 |-- unit_terjual: integer (nullable = true)
 |-- tanggal: timestamp (nullable = true)

Data Produk: 30 baris
root
 |-- harga: long (nullable = true)
 |-- kategori: string (nullable = true)
 |-- nama_produk: string (nullable = true)
 |-- product_id: long (nullable = true)

Data Ulasan: 3500 baris
root
 |-- order_id: string (nullable = true)
 |-- rating: integer (nullable = true)



In [8]:
from pyspark.sql.functions import col

# B. TRANSFORM - JOIN
# Join Transaksi ke Produk menggunakan INNER JOIN (karena setiap transaksi pasti ada produknya)
df_gabung_1 = df_transaksi.join(df_produk, on="product_id", how="inner")

# Join hasilnya ke Ulasan menggunakan LEFT JOIN (karena tidak semua transaksi memiliki ulasan)
df_gabung_2 = df_gabung_1.join(df_ulasan, on="order_id", how="left")

# Menambahkan kolom total_pendapatan (unit_terjual * harga)
df_gabung_2 = df_gabung_2.withColumn("total_pendapatan", col("unit_terjual") * col("harga"))

# Menampilkan 5 baris pertama untuk mengecek hasil join
df_gabung_2.show(5)

# Mengecek total baris setelah join
print("Total baris setelah digabung:", df_gabung_2.count())

+--------+----------+------------+-------------------+------+------------+-----------+------+----------------+
|order_id|product_id|unit_terjual|            tanggal| harga|    kategori|nama_produk|rating|total_pendapatan|
+--------+----------+------------+-------------------+------+------------+-----------+------+----------------+
|     TX0|        21|           2|2026-10-10 00:00:00| 25000|  Elektronik|  Produk-21|     1|           50000|
|     TX1|         8|           6|2026-10-25 00:00:00|250000|     Fashion|   Produk-8|  NULL|         1500000|
|     TX2|        25|           4|2026-10-13 00:00:00| 25000|  Elektronik|  Produk-25|     2|          100000|
|     TX3|         3|           4|2026-10-18 00:00:00| 75000|     Fashion|   Produk-3|     5|          300000|
|     TX4|        19|           6|2026-10-07 00:00:00| 75000|Rumah Tangga|  Produk-19|  NULL|          450000|
+--------+----------+------------+-------------------+------+------------+-----------+------+----------------+
o

In [9]:
# C. TRANSFORM - DATA QUALITY
# 1. Tambah kolom 'ada_ulasan' (True jika rating tidak null, False jika null) SEBELUM na.fill()
df_transform = df_gabung_2.withColumn("ada_ulasan", col("rating").isNotNull())

# 2. Mengisi nilai rating yang kosong (null) dengan angka 0
df_transform = df_transform.fillna({"rating": 0})

df_transform.select("order_id", "nama_produk", "unit_terjual", "rating", "ada_ulasan", "total_pendapatan").show(5)

+--------+-----------+------------+------+----------+----------------+
|order_id|nama_produk|unit_terjual|rating|ada_ulasan|total_pendapatan|
+--------+-----------+------------+------+----------+----------------+
|     TX0|  Produk-21|           2|     1|      true|           50000|
|     TX1|   Produk-8|           6|     0|     false|         1500000|
|     TX2|  Produk-25|           4|     2|      true|          100000|
|     TX3|   Produk-3|           4|     5|      true|          300000|
|     TX4|  Produk-19|           6|     0|     false|          450000|
+--------+-----------+------------+------+----------+----------------+
only showing top 5 rows



In [11]:
# D. LOAD
path_hdfs = "hdfs://localhost:9000/user/mahasiswa/tugas6/hasil_etl"

# Membuat direktori jika belum ada (opsional tapi disarankan)
!hdfs dfs -mkdir -p /user/mahasiswa/tugas6/

# Menyimpan ke HDFS dengan format Parquet, dipartisi berdasarkan 'kategori'
df_transform.write.mode("overwrite").partitionBy("kategori").parquet(path_hdfs)

# Memverifikasi struktur folder di HDFS
print("Struktur Folder di HDFS:")
!hdfs dfs -ls -R /user/mahasiswa/tugas6/hasil_etl

# Membaca kembali dari HDFS untuk memastikan data tersimpan utuh
df_final = spark.read.parquet(path_hdfs)
print("\nJumlah baris hasil ETL yang berhasil dimuat dari HDFS:", df_final.count())

Struktur Folder di HDFS:
-rw-r--r--   3 vandrasembiring supergroup          0 2026-09-30 21:29 /user/mahasiswa/tugas6/hasil_etl/_SUCCESS
drwxr-xr-x   - vandrasembiring supergroup          0 2026-09-30 21:29 /user/mahasiswa/tugas6/hasil_etl/kategori=Elektronik
-rw-r--r--   3 vandrasembiring supergroup      16364 2026-09-30 21:29 /user/mahasiswa/tugas6/hasil_etl/kategori=Elektronik/part-00000-7358e464-c089-4ff9-956b-22a08aece36c.c000.snappy.parquet
drwxr-xr-x   - vandrasembiring supergroup          0 2026-09-30 21:29 /user/mahasiswa/tugas6/hasil_etl/kategori=Fashion
-rw-r--r--   3 vandrasembiring supergroup      11134 2026-09-30 21:29 /user/mahasiswa/tugas6/hasil_etl/kategori=Fashion/part-00000-7358e464-c089-4ff9-956b-22a08aece36c.c000.snappy.parquet
drwxr-xr-x   - vandrasembiring supergroup          0 2026-09-30 21:29 /user/mahasiswa/tugas6/hasil_etl/kategori=Kesehatan
-rw-r--r--   3 vandrasembiring supergroup      10393 2026-09-30 21:29 /user/mahasiswa/tugas6/hasil_etl/kategori=Kesehat

In [12]:
from pyspark.sql.functions import avg, round

# E. INSIGHT AKHIR
# Menghitung persentase ulasan. Karena 'ada_ulasan' bernilai boolean (True=1, False=0), 
# rata-ratanya dikali 100 langsung menghasilkan persentase.
df_insight = df_final.groupBy("kategori") \
    .agg(round(avg(col("ada_ulasan").cast("int")) * 100, 2).alias("persentase_diulas")) \
    .orderBy("persentase_diulas")

df_insight.show()

+------------+-----------------+
|    kategori|persentase_diulas|
+------------+-----------------+
|     Makanan|            68.84|
|   Kesehatan|            68.89|
|     Fashion|            70.14|
|Rumah Tangga|            70.55|
|  Elektronik|            70.66|
+------------+-----------------+

